### 📝 Notas de Documentación - Capa Bronze (USGS Earthquakes)

- **Fuente:** USGS (United States Geological Survey).
- **Endpoint consumido:** `https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_day.geojson`[cite: 6].
- **Filtro de Ingesta:** Sismos de **magnitud 2.5 o superior** ocurridos en las **últimas 24 horas** a nivel mundial[cite: 6].
- **Volumen ingerido:** 38 registros como primer obtencion
- **Frecuencia de refresco recomendada:** Ejecución diaria o por hora mediante Airflow[cite: 6].
- **Resguardo Crudo:** Guardado en la Landing Zone (`stack/data/raw/usgs_earthquakes/ds=YYYY-MM-DD/`) como archivo JSON inmutable para auditoría y reproducibilidad (*replay*)[cite: 6].

In [11]:
import json
import os
import subprocess
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import requests
from sqlalchemy import create_engine


# 1. Hallar la raíz del proyecto dinámicamente mediante Git
def get_repo_root() -> Path:
  try:
    return Path(
        subprocess.check_output(
            ['git', 'rev-parse', '--show-toplevel'], stderr=subprocess.DEVNULL
        )
        .decode()
        .strip()
    )
  except Exception:
    # Si ejecutas el notebook dentro de notebooks/ o grupos/G12/notebooks/
    return Path('../../..')


# 2. Definir rutas relativas universales
REPO_ROOT = get_repo_root()

# Cargar el archivo .env desde tu carpeta Stack
dotenv_path = REPO_ROOT / 'TpFinal' / 'grupos' / 'G12' / 'Stack' / '.env'
if not dotenv_path.exists():
  # Intentar buscarlo directamente en la carpeta Stack donde estás parado
  dotenv_path = Path('.env')

load_dotenv(dotenv_path=dotenv_path)

# 3. Leer credenciales de Postgres desde tu .env cargado
DB_USER = os.getenv('SOURCE_DB_USER', 'admin')  #[cite: 4]
DB_PASS = os.getenv('SOURCE_DB_PASS', 'admin')  #[cite: 4]
DB_NAME = os.getenv('SOURCE_DB_NAME', 'InfraCienciaDatos')  #[cite: 4]
DB_HOST = 'localhost'
DB_PORT = 5432

# 4. Instanciar motor de conexión
DATABASE_URL = (
    f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)
engine = create_engine(DATABASE_URL)

# 5. Ruta para resguardar los JSON crudos en la Landing Zone (Data Lake)
# Mapeado a: G12/Stack/data/raw/usgs_earthquakes/
CARPETA_RAW = dotenv_path.parent / 'data' / 'raw' / 'usgs_earthquakes'

print(f'✅ Conexión configurada a Postgres DW: {DB_NAME}')
print(f'📁 Carpeta Landing Zone (Data Lake): {CARPETA_RAW}')

✅ Conexión configurada a Postgres DW: InfraCienciaDatos
📁 Carpeta Landing Zone (Data Lake): C:\Users\Usuario\Documents\Facultad\3_Infra\LCD-InfraCienciaDatos\TpFinal\grupos\G12\Stack\data\raw\usgs_earthquakes


Traigo los datos en bronze

In [12]:
def ingestar_usgs_bronze(modo='replace'):
  url_api = (
      'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_day.geojson'
  )

  # 1. Petición HTTP a la API
  response = requests.get(url_api)
  response.raise_for_status()
  ahora = datetime.now()

  # 2. Guardar JSON crudo en la Landing Zone / Data Lake local
  carpeta_particion = CARPETA_RAW / f'ds={ahora:%Y-%m-%d}'
  carpeta_particion.mkdir(parents=True, exist_ok=True)
  archivo_json = carpeta_particion / f'{ahora:%H%M%S_%f}.json'
  archivo_json.write_bytes(response.content)

  # 3. Formatear la lista GeoJSON ('features')
  payload = response.json()
  features = payload.get('features', [])

  df = pd.DataFrame(features)

  # Serializar estructuras internas (properties, geometry) a string de JSON
  for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
      df[col] = df[col].apply(
          lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
      )

  # Auditoría Bronze
  df['ingested_at'] = ahora

  # 4. Guardar en Postgres usando str(engine.url) para asegurar compatibilidad
  df.to_sql(
      name='raw_earthquakes',
      con=str(engine.url),
      schema='bronze',
      if_exists=modo,
      index=False,
  )

  print(f'✅ Ingesta exitosa ({ahora:%H:%M:%S} - Modo: {modo})')
  print(f' 📄 Archivo Raw: {archivo_json.name}')
  print(f' 🗄️ Tabla Postgres: bronze.raw_earthquakes (+{len(df)} filas)')


# Ejecutar ingesta
ingestar_usgs_bronze(modo='replace')

✅ Ingesta exitosa (10:29:12 - Modo: replace)
 📄 Archivo Raw: 102912_455090.json
 🗄️ Tabla Postgres: bronze.raw_earthquakes (+38 filas)


In [13]:
with engine.connect() as conn:
  df_verificacion = pd.read_sql(
      'SELECT id, type, ingested_at FROM bronze.raw_earthquakes LIMIT 5;',
      con=conn,
  )

df_verificacion

,id,type,ingested_at
0,aka2026whbwrl,Feature,2026-11-10 10:29:12.455090
1,aka2026wgwxww,Feature,2026-11-10 10:29:12.455090
2,pr2026314000,Feature,2026-11-10 10:29:12.455090
3,us7000tx9z,Feature,2026-11-10 10:29:12.455090
4,uw714136981,Feature,2026-11-10 10:29:12.455090


In [14]:
import pandas as pd

# 1. Leer la tabla bronze directamente desde PostgreSQL
with engine.connect() as conn:
  df_bronze = pd.read_sql('SELECT * FROM bronze.raw_earthquakes;', con=conn)

# 2. Ver la cantidad de filas y columnas devueltas
print(f'Total de registros cargados: {len(df_bronze)}')

# 3. Mostrar la tabla interactiva en Jupyter
df_bronze.head(10)

Total de registros cargados: 38


,type,properties,geometry,id,ingested_at
0,Feature,"{""mag"": 2.6, ""place"": ""51 km N of Aleneva, Ala...","{""type"": ""Point"", ""coordinates"": [-152.996, 58...",aka2026whbwrl,2026-11-10 10:29:12.455090
1,Feature,"{""mag"": 2.5, ""place"": ""59 km E of Pedro Bay, A...","{""type"": ""Point"", ""coordinates"": [-153.067, 59...",aka2026wgwxww,2026-11-10 10:29:12.455090
2,Feature,"{""mag"": 3.68, ""place"": ""69 km N of Culebra, Pu...","{""type"": ""Point"", ""coordinates"": [-65.2388, 18...",pr2026314000,2026-11-10 10:29:12.455090
3,Feature,"{""mag"": 5, ""place"": ""58 km NE of Fef, Indonesi...","{""type"": ""Point"", ""coordinates"": [132.859, -0....",us7000tx9z,2026-11-10 10:29:12.455090
4,Feature,"{""mag"": 4.29, ""place"": ""19 km ESE of Joseph, O...","{""type"": ""Point"", ""coordinates"": [-117.0203333...",uw714136981,2026-11-10 10:29:12.455090
5,Feature,"{""mag"": 4.8, ""place"": ""northern Mid-Atlantic R...","{""type"": ""Point"", ""coordinates"": [-35.075, 34....",us7000tx9n,2026-11-10 10:29:12.455090
6,Feature,"{""mag"": 2.89, ""place"": ""75 km NNE of Vieques, ...","{""type"": ""Point"", ""coordinates"": [-65.51133333...",pr71538963,2026-11-10 10:29:12.455090
7,Feature,"{""mag"": 4.7, ""place"": ""90 km SSE of Tateyama, ...","{""type"": ""Point"", ""coordinates"": [140.1377, 34...",us7000tx9i,2026-11-10 10:29:12.455090
8,Feature,"{""mag"": 3.41, ""place"": ""11 km NE of Boca de Yu...","{""type"": ""Point"", ""coordinates"": [-68.5245, 18...",pr71538948,2026-11-10 10:29:12.455090
9,Feature,"{""mag"": 4.9, ""place"": ""78 km ESE of Hitachi, J...","{""type"": ""Point"", ""coordinates"": [141.4659, 36...",us7000tx96,2026-11-10 10:29:12.455090


Ingesta completada pasamos a la capa silver.

In [30]:
import json
import pandas as pd

# 1. Leer los datos crudos desde la Capa Bronze en PostgreSQL
with engine.connect() as conn:
  df_bronze = pd.read_sql('SELECT * FROM bronze.raw_earthquakes;', con=conn)


# 2. Función de transformación Silver
def procesar_silver_usgs(df_raw):
  registros = []

  for idx, row in df_raw.iterrows():
    # Parsear campos JSON en caso de que vengan como strings
    props = (
        json.loads(row['properties'])
        if isinstance(row['properties'], str)
        else row['properties']
    )
    geom = (
        json.loads(row['geometry'])
        if isinstance(row['geometry'], str)
        else row['geometry']
    )

    # Extraer coordenadas [longitud, latitud, profundidad] de geometry
    coords = geom.get('coordinates', [None, None, None])
    lon, lat, depth = coords[0], coords[1], coords[2]

    # Magnitud y Cálculo de Energía Liberada (Joules)
    mag = props.get('mag')
    if mag is not None and mag > 0:
      energia_joules = 10 ** (4.8 + 1.5 * mag)
    else:
      energia_joules = None

    # Categorización por Escala de Severidad
    if mag is None:
      escala = 'Desconocida'
    elif mag < 3.0:
      escala = 'Menor'
    elif mag < 4.5:
      escala = 'Leve'
    elif mag < 6.0:
      escala = 'Moderado'
    else:
      escala = 'Fuerte / Mayor'

    # Separa localidad y región/país desde el campo 'place'
    place_str = props.get('place') or ''
    if ',' in place_str:
      partes = place_str.split(',')
      detalles_localidad = partes[0].strip()
      region = partes[1].strip()
    else:
      detalles_localidad = place_str
      region = place_str

    # Convertir timestamp UNIX (ms) a Datetime UTC
    time_ms = props.get('time')
    event_dt = pd.to_datetime(time_ms, unit='ms') if time_ms else None

    registros.append({
        'earthquake_id': row['id'],
        'magnitude': mag,
        'escala_magnitud': escala,
        'energia_joules': energia_joules,
        'place': place_str,
        'place_detail': detalles_localidad,
        'region': region,
        'tsunami_alert': int(props.get('tsunami', 0)),
        'event_timestamp': event_dt,
        'event_date': event_dt.date() if event_dt else None,
        'longitude': lon,
        'latitude': lat,
        'depth_km': depth,
        'ingested_at': row['ingested_at'],
        'processed_at': pd.Timestamp.now(),
    })

  df_silver = pd.DataFrame(registros)

  # 3. Deduplicación por la PK del sismo (Regla esencial de Silver)
  df_silver = df_silver.drop_duplicates(subset=['earthquake_id'])

  return df_silver


# 3. Ejecutar transformación y persistir en la base de datos
df_silver = procesar_silver_usgs(df_bronze)

df_silver.to_sql(
    name='earthquakes',
    con=str(engine.url),
    schema='silver',
    if_exists='replace',
    index=False,
)

print(
    f'🥈 Capa Silver cargada exitosamente: {len(df_silver)} sismos limpios y'
    ' enriquecidos.'
)

🥈 Capa Silver cargada exitosamente: 38 sismos limpios y enriquecidos.


In [28]:
with engine.connect() as conn:
  df_check = pd.read_sql(
      'SELECT * FROM silver.earthquakes LIMIT 6',
      con=conn,
  )

df_check

,earthquake_id,magnitud,escala_magnitud,energia_joules,place,place_detail,region,tsunami_alert,event_timestamp,event_date,longitude,latitude,depth_km,ingested_at,processed_at
0,aka2026whbwrl,2.60,Menor,5.011872e+08,"51 km N of Aleneva, Alaska",51 km N of Aleneva,Alaska,0,2026-11-10 12:30:36.703,2026-11-10,-152.996000,58.516000,64.30,2026-11-10 10:29:12.455090,2026-11-10 11:35:04.177549
1,aka2026wgwxww,2.50,Menor,3.548134e+08,"59 km E of Pedro Bay, Alaska",59 km E of Pedro Bay,Alaska,0,2026-11-10 10:02:02.815,2026-11-10,-153.067000,59.696000,111.10,2026-11-10 10:29:12.455090,2026-11-10 11:35:04.177549
2,pr2026314000,3.68,Leve,2.089296e+10,"69 km N of Culebra, Puerto Rico",69 km N of Culebra,Puerto Rico,0,2026-11-10 08:48:04.860,2026-11-10,-65.238800,18.928500,14.00,2026-11-10 10:29:12.455090,2026-11-10 11:35:04.177549
3,us7000tx9z,5.00,Moderado,1.995262e+12,"58 km NE of Fef, Indonesia",58 km NE of Fef,Indonesia,0,2026-11-10 08:34:07.131,2026-11-10,132.859000,-0.492800,35.00,2026-11-10 10:29:12.455090,2026-11-10 11:35:04.180601
4,uw714136981,4.29,Leve,1.717908e+11,"19 km ESE of Joseph, Oregon",19 km ESE of Joseph,Oregon,0,2026-11-10 07:42:52.710,2026-11-10,-117.020333,45.267667,2.52,2026-11-10 10:29:12.455090,2026-11-10 11:35:04.180601
5,us7000tx9n,4.80,Moderado,1.000000e+12,northern Mid-Atlantic Ridge,northern Mid-Atlantic Ridge,northern Mid-Atlantic Ridge,0,2026-11-10 06:43:11.261,2026-11-10,-35.075000,34.877400,10.00,2026-11-10 10:29:12.455090,2026-11-10 11:35:04.180601


### 🥈 Capa Silver — Curación, Desanidado y Enriquecimiento (`silver.earthquakes`)

En esta etapa transformamos los datos crudos almacenados en `bronze.raw_earthquakes` a un modelo relacional limpio, estandarizado y deduplicado.

#### 🎯 Objetivos de la Capa Silver:
1. **Desanidado de GeoJSON:** 
   - Aplanado de la columna `geometry`: extracción de `longitude`, `latitude` y `depth_km` a partir de la lista de coordenadas.
   - Desestructuración de la columna `properties`: extracción de magnitud, ubicación, timestamp y alertas.
2. **Normalización Geográfica y Temporal:**
   - Limpieza del campo `place` para separar el detalle específico del sismo de la `region`/país principal.
   - Conversión de timestamps UNIX (expresados en milisegundos por USGS) a tipos de datos relacionales `TIMESTAMP` UTC y `DATE`.
3. **Feature Engineering (Dominio de Negocio):**
   - **Categorización Sísmica:** Clasificación cualitativa de la escala (`Menor`, `Leve`, `Moderado`, `Fuerte / Mayor`).
   - **Energía Liberada ($E$):** Cálculo de la energía en Joules mediante la fórmula empírica sísmica:
     $$E = 10^{4.8 + 1.5 \cdot \text{magnitud}}$$
4. **Calidad y Deduplicación:**
   - Filtrado y deduplicación estricta tomando como clave primaria el `earthquake_id` para garantizar idempotencia.
   - Adición de metadatos de trazabilidad (`processed_at`).

- **Tabla Destino:** `silver.earthquakes` (PostgreSQL)

A continuacion seguimos con la capa gold.

In [31]:
import pandas as pd
from sqlalchemy import create_engine

# -------------------------------------------------------------------------
# 1. Leer los datos curados y limpios desde la Capa Silver (PostgreSQL)
# -------------------------------------------------------------------------
with engine.connect() as conn:
  df_silver = pd.read_sql('SELECT * FROM silver.earthquakes;', con=conn)

print(
    f'📥 Leídos {len(df_silver)} registros desde silver.earthquakes para'
    ' procesar Gold.'
)

# -------------------------------------------------------------------------
# 2. Dimensión Región (gold.dim_region)
# -------------------------------------------------------------------------
df_dim_region = df_silver[['region']].drop_duplicates().reset_index(drop=True)
df_dim_region['region_id'] = df_dim_region.index + 1
df_dim_region = df_dim_region[['region_id', 'region']]

# Unir surrogate key (region_id) a los datos base de sismos
df_base_gold = df_silver.merge(df_dim_region, on='region')

# -------------------------------------------------------------------------
# 3. Tabla de Hechos (gold.fact_sismos_diarios)
# Granularidad: Fecha de evento + Región
# -------------------------------------------------------------------------
df_fact_sismos_diarios = (
    df_base_gold.groupby(['event_date', 'region_id'])
    .agg(
        cantidad_sismos=('earthquake_id', 'count'),
        magnitud_promedio=('magnitude', 'mean'),
        magnitud_maxima=('magnitude', 'max'),
        energia_total_joules=('energia_joules', 'sum'),
        alertas_tsunami=('tsunami_alert', 'sum'),
    )
    .reset_index()
)

# Rounding de decimales para presentación analítica
df_fact_sismos_diarios['magnitud_promedio'] = df_fact_sismos_diarios[
    'magnitud_promedio'
].round(2)
df_fact_sismos_diarios['magnitud_maxima'] = df_fact_sismos_diarios[
    'magnitud_maxima'
].round(2)

# -------------------------------------------------------------------------
# 4. Data Mart / Ranking de Zonas Críticas (gold.agg_ranking_zonas)
# Responde directo al Dashboard: ¿Dónde ocurren más sismos?
# -------------------------------------------------------------------------
df_ranking_zonas = (
    df_base_gold.groupby('region')
    .agg(
        total_sismos=('earthquake_id', 'count'),
        magnitud_promedio=('magnitude', 'mean'),
        magnitud_maxima=('magnitude', 'max'),
        profundidad_promedio_km=('depth_km', 'mean'),
        total_alertas_tsunami=('tsunami_alert', 'sum'),
    )
    .reset_index()
    .sort_values(by='total_sismos', ascending=False)
)

df_ranking_zonas['magnitud_promedio'] = df_ranking_zonas[
    'magnitud_promedio'
].round(2)
df_ranking_zonas['magnitud_maxima'] = df_ranking_zonas['magnitud_maxima'].round(
    2
)
df_ranking_zonas['profundidad_promedio_km'] = df_ranking_zonas[
    'profundidad_promedio_km'
].round(2)

# -------------------------------------------------------------------------
# 5. Persistir las 3 tablas en el esquema 'gold' de PostgreSQL
# -------------------------------------------------------------------------
df_dim_region.to_sql(
    name='dim_region',
    con=str(engine.url),
    schema='gold',
    if_exists='replace',
    index=False,
)
df_fact_sismos_diarios.to_sql(
    name='fact_sismos_diarios',
    con=str(engine.url),
    schema='gold',
    if_exists='replace',
    index=False,
)
df_ranking_zonas.to_sql(
    name='agg_ranking_zonas',
    con=str(engine.url),
    schema='gold',
    if_exists='replace',
    index=False,
)

print('\n🥇 Capa Gold procesada exitosamente en PostgreSQL:')
print(f' ▫️ gold.dim_region: {len(df_dim_region)} filas')
print(f' ▫️ gold.fact_sismos_diarios: {len(df_fact_sismos_diarios)} filas')
print(f' ▫️ gold.agg_ranking_zonas: {len(df_ranking_zonas)} filas')

📥 Leídos 38 registros desde silver.earthquakes para procesar Gold.

🥇 Capa Gold procesada exitosamente en PostgreSQL:
 ▫️ gold.dim_region: 18 filas
 ▫️ gold.fact_sismos_diarios: 23 filas
 ▫️ gold.agg_ranking_zonas: 18 filas


In [32]:
with engine.connect() as conn:
  df_top_zonas = pd.read_sql(
      'SELECT * FROM gold.agg_ranking_zonas LIMIT 5;', con=conn
  )

df_top_zonas

,region,total_sismos,magnitud_promedio,magnitud_maxima,profundidad_promedio_km,total_alertas_tsunami
0,Alaska,10,3.12,4.70,41.12,0
1,Puerto Rico,3,3.19,3.68,21.13,0
2,Dominican Republic,3,3.35,3.63,47.00,0
3,Indonesia,3,4.90,5.40,18.33,0
4,Solomon Islands,3,4.80,5.20,10.00,0


### 🥇 Capa Gold — Modelo Dimensional y Data Mart (`gold.*`)

En esta etapa construimos el **Modelo Estrella (Star Schema)** y las agregaciones de negocio para alimentar de forma directa el Dashboard interactivo de Streamlit.

#### 📐 Componentes del Modelo:
1. **`gold.dim_region` (Dimensión):**
   - Tabla de dimensión geográfica con clave primaria subrogada (`region_id`) y catálogo único de regiones.
2. **`gold.fact_sismos_diarios` (Tabla de Hechos):**
   - Granularidad: **Fecha de Evento + Región**.
   - Métricas: Cantidad de sismos, magnitud promedio, magnitud máxima, energía total liberada (Joules) y alertas de tsunami emitidas.
3. **`gold.agg_ranking_zonas` (Data Mart de Consumo):**
   - Agregación orientada a responder las preguntas del Dashboard: identifica las regiones con mayor frecuencia de eventos y compara profundidades y severidades.